In [ ]:
# Multi-model evaluation on processed BraTS test set with segmentation-derived labels
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    auc,
 )

# =========================
# ENV / PATHS
# =========================
try:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
except Exception:
    pass

REPO_ROOT = Path('/content/symAD-ECNN')
if not REPO_ROOT.exists():
    # optional clone if needed
    import subprocess
    subprocess.check_call(['git', 'clone', 'https://github.com/RifaDeen/symAD-ECNN.git', str(REPO_ROOT)])

EVALS_DIR = REPO_ROOT / 'notebooks' / 'evals'
ECNN_EVAL_DIR = EVALS_DIR / 'ecnn_thresholding'
for p in [REPO_ROOT, EVALS_DIR, ECNN_EVAL_DIR]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from ecnn_model_loader import get_model_for_inference

PROJECT_ROOT = Path('/content/drive/MyDrive/symAD-ECNN')
MODELS_ROOT = PROJECT_ROOT / 'models' / 'saved_models'
PROCESSED_ROOT = Path('/content/test_eval_data/processed')
IMG_DIR = PROCESSED_ROOT / 'images'
LBL_DIR = PROCESSED_ROOT / 'labels'

OUT_CSV = PROJECT_ROOT / 'evaluations' / 'tables' / 'test_with_masks_metrics_fpr20.csv'
OUT_FIG_DIR = PROJECT_ROOT / 'evaluations' / 'figures' / 'test_with_masks'
OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

TARGET_FPR = 0.20
BATCH_SIZE = 32
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'Processed data: {PROCESSED_ROOT}')
print(f'Model root: {MODELS_ROOT}')

# =========================
# MODEL DEFINITIONS
# =========================
class CNNAutoencoder(nn.Module):
    def __init__(self, latent_dim: int = 256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1, 1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2,2),
            nn.Conv2d(32, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2,2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2,2),
            nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True), nn.MaxPool2d(2,2),
        )
        self.flatten = nn.Flatten()
        self.fc_encode = nn.Linear(8 * 8 * 256, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 8 * 8 * 256)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 256, 3, 2, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 3, 2, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 3, 2, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, 3, 2, 1, 1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 1, 3, 1, 1), nn.Sigmoid(),
        )
    def forward(self, x):
        feats = self.encoder(x)
        b = feats.size(0)
        z = self.fc_encode(self.flatten(feats))
        dec = self.fc_decode(z).view(b, 256, 8, 8)
        return self.decoder(dec)

class LargeCNNAutoencoder(nn.Module):
    def __init__(self, latent_dim: int = 512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2,2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2,2),
            nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True), nn.MaxPool2d(2,2),
            nn.Conv2d(256, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(True), nn.MaxPool2d(2,2),
        )
        self.flatten = nn.Flatten()
        self.fc_encode = nn.Linear(8 * 8 * 512, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 8 * 8 * 512)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 512, 3, 2, 1, 1), nn.BatchNorm2d(512), nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 3, 2, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 3, 2, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 3, 2, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 1, 3, 1, 1), nn.Sigmoid(),
        )
    def forward(self, x):
        feats = self.encoder(x)
        b = feats.size(0)
        z = self.fc_encode(self.flatten(feats))
        dec = self.fc_decode(z).view(b, 512, 8, 8)
        return self.decoder(dec)

class ResNetAutoencoder(nn.Module):
    def __init__(self, finetune_strategy='none'):
        super().__init__()
        resnet = models.resnet18(weights=None)
        self.encoder = nn.Sequential(*list(resnet.children())[:-1])

        if finetune_strategy == 'none':
            for p in self.encoder.parameters():
                p.requires_grad = False
        elif finetune_strategy == 'partial':
            for i, module in enumerate(self.encoder.children()):
                train = i >= 7
                for p in module.parameters():
                    p.requires_grad = train
        elif finetune_strategy == 'full':
            for p in self.encoder.parameters():
                p.requires_grad = True

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 512, 4, 1, 0), nn.BatchNorm2d(512), nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.ConvTranspose2d(32, 1, 4, 2, 1), nn.Sigmoid(),
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

# =========================
# DATASET
# =========================
class EvalDataset(Dataset):
    def __init__(self, image_dir: Path, label_dir: Path, mode='grayscale'):
        self.image_files = sorted(image_dir.glob('slice_*.npy'))
        self.label_dir = label_dir
        self.mode = mode
        self.resnet_transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        sid = img_path.stem.split('_')[-1]
        lab_path = self.label_dir / f'label_{sid}.npy'

        img = np.load(img_path).astype(np.float32)
        label = int(np.load(lab_path)[0])

        gray = torch.from_numpy(img).float().unsqueeze(0)
        target = gray.clone()

        if self.mode == 'resnet':
            img_uint8 = (img * 255.0).clip(0, 255).astype(np.uint8)
            rgb = np.stack([img_uint8, img_uint8, img_uint8], axis=-1)
            inp = self.resnet_transform(rgb)
            return inp, target, label
        return gray, target, label

# =========================
# LOADERS / METRICS
# =========================
def get_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if 'model_state_dict' in ckpt and isinstance(ckpt['model_state_dict'], dict):
            return ckpt['model_state_dict']
        if 'state_dict' in ckpt and isinstance(ckpt['state_dict'], dict):
            return ckpt['state_dict']
    return ckpt

def find_ckpt(subdirs, patterns):
    for sd in subdirs:
        root = MODELS_ROOT / sd if sd != '.' else MODELS_ROOT
        if not root.exists():
            continue
        for pat in patterns:
            ms = sorted(root.rglob(pat))
            if ms:
                return ms[0]
    return None

MODEL_CONFIGS = [
    {
        'name': 'ECNN-AE (Optimized)',
        'key': 'ecnn_opt',
        'type': 'ecnn',
        'subdirs': ['ecnn_optimized', '.'],
        'patterns': ['ecnn_optimized_best.pth', 'ecnn_best.pth', '*ecnn*best*.pth'],
    },
    {
        'name': 'CNN-AE Large',
        'key': 'cnn_large',
        'type': 'cnn_large',
        'subdirs': ['cnn_ae_large'],
        'patterns': ['cnn_large_best.pth', '*cnn*large*best*.pth'],
    },
    {
        'name': 'CNN-AE Augmented',
        'key': 'cnn_aug',
        'type': 'cnn_base',
        'subdirs': ['cnn_ae_augmented'],
        'patterns': ['cnn_aug_best.pth', '*cnn*aug*best*.pth'],
    },
    {
        'name': 'CNN-AE',
        'key': 'cnn_base',
        'type': 'cnn_base',
        'subdirs': ['cnn_ae'],
        'patterns': ['cnn_best.pth', '*cnn*best*.pth'],
    },
    {
        'name': 'ResNet-AE',
        'key': 'resnet_ae',
        'type': 'resnet_frozen',
        'subdirs': ['resnet_ae'],
        'patterns': ['resnet_best.pth', '*resnet*best*.pth'],
    },
    {
        'name': 'ResNet Finetuned (partial)',
        'key': 'resnet_ft',
        'type': 'resnet_ft_partial',
        'subdirs': ['resnet_finetuned_partial', 'resnet_finetuned_full', 'resnet_finetuned_none'],
        'patterns': ['resnet_best.pth', '*resnet*best*.pth'],
    },
]

def load_model(cfg):
    ckpt_path = find_ckpt(cfg['subdirs'], cfg['patterns'])
    if ckpt_path is None:
        raise FileNotFoundError(f"Checkpoint not found for {cfg['name']}")

    if cfg['type'] == 'ecnn':
        model, _ = get_model_for_inference(ckpt_path, str(device))
        return model.to(device).eval(), ckpt_path

    ckpt = torch.load(ckpt_path, map_location=device)
    sd = get_state_dict(ckpt)

    if cfg['type'] == 'cnn_large':
        latent = int(sd['fc_encode.weight'].shape[0]) if 'fc_encode.weight' in sd else 512
        model = LargeCNNAutoencoder(latent_dim=latent)
    elif cfg['type'] == 'cnn_base':
        latent = int(sd['fc_encode.weight'].shape[0]) if 'fc_encode.weight' in sd else 256
        model = CNNAutoencoder(latent_dim=latent)
    elif cfg['type'] == 'resnet_frozen':
        model = ResNetAutoencoder(finetune_strategy='none')
    elif cfg['type'] == 'resnet_ft_partial':
        model = ResNetAutoencoder(finetune_strategy='partial')
    else:
        raise ValueError(cfg['type'])

    model.load_state_dict(sd, strict=False)
    return model.to(device).eval(), ckpt_path

@torch.no_grad()
def infer_scores(model, dataloader):
    scores, labels = [], []
    for x, target, lab in tqdm(dataloader, leave=False):
        x = x.to(device)
        target = target.to(device)
        recon = model(x)
        mse = F.mse_loss(recon, target, reduction='none').view(x.size(0), -1).mean(dim=1)
        scores.extend(mse.detach().cpu().numpy().tolist())
        labels.extend(lab.numpy().tolist())
    return np.asarray(scores, dtype=np.float32), np.asarray(labels, dtype=np.int32)

def threshold_from_normals(scores, labels, target_fpr=0.20):
    normal_scores = scores[labels == 0]
    if len(normal_scores) == 0:
        raise RuntimeError('No normal samples found (label=0).')
    return float(np.percentile(normal_scores, (1 - target_fpr) * 100))

def compute_metrics(labels, scores, threshold):
    preds = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1 = f1_score(labels, preds, zero_division=0)
    acc = accuracy_score(labels, preds)
    auroc = roc_auc_score(labels, scores) if len(np.unique(labels)) > 1 else 0.5
    auprc = average_precision_score(labels, scores) if len(np.unique(labels)) > 1 else 0.0
    return {
        'threshold': threshold,
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'specificity': specificity,
        'f1_score': f1,
        'fpr': fp / (fp + tn) if (fp + tn) > 0 else 0.0,
        'fnr': fn / (fn + tp) if (fn + tp) > 0 else 0.0,
        'auroc': auroc,
        'auprc': auprc,
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn),
    }

def plot_and_save_cm(labels, scores, threshold, title_prefix, key):
    preds = (scores >= threshold).astype(int)
    cm = confusion_matrix(labels, preds, labels=[0,1])
    fig, ax = plt.subplots(1,2, figsize=(12,5))
    im0 = ax[0].imshow(cm, cmap='Blues')
    ax[0].set_title(f'{title_prefix} - CM (counts)')
    ax[0].set_xticks([0,1]); ax[0].set_yticks([0,1]); ax[0].set_xticklabels(['Normal','Anomaly']); ax[0].set_yticklabels(['Normal','Anomaly'])
    for i in range(2):
        for j in range(2):
            ax[0].text(j, i, str(cm[i,j]), ha='center', va='center')
    fig.colorbar(im0, ax=ax[0], fraction=0.046)

    cmn = cm.astype(np.float32) / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    im1 = ax[1].imshow(cmn, cmap='Blues', vmin=0, vmax=1)
    ax[1].set_title(f'{title_prefix} - CM (normalized)')
    ax[1].set_xticks([0,1]); ax[1].set_yticks([0,1]); ax[1].set_xticklabels(['Normal','Anomaly']); ax[1].set_yticklabels(['Normal','Anomaly'])
    for i in range(2):
        for j in range(2):
            ax[1].text(j, i, f'{cmn[i,j]:.2f}', ha='center', va='center')
    fig.colorbar(im1, ax=ax[1], fraction=0.046)
    plt.tight_layout()
    fig.savefig(OUT_FIG_DIR / f'{key}_confusion_matrices.png', dpi=150)
    plt.close(fig)

def plot_and_save_roc(labels, scores, title_prefix, key):
    fpr, tpr, _ = roc_curve(labels, scores)
    auroc = roc_auc_score(labels, scores) if len(np.unique(labels)) > 1 else 0.5
    fig, ax = plt.subplots(figsize=(6,5))
    ax.plot(fpr, tpr, lw=2, label=f'AUROC={auroc:.4f}')
    ax.plot([0,1],[0,1],'k--',lw=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'{title_prefix} - ROC')
    ax.legend(loc='lower right')
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT_FIG_DIR / f'{key}_roc.png', dpi=150)
    plt.close(fig)
    return fpr, tpr, auroc

# =========================
# RUN EVALUATION
# =========================
if not IMG_DIR.exists() or not LBL_DIR.exists():
    raise FileNotFoundError('Processed dataset not found. Run test_preprocessing notebook first.')

rows = []
roc_curves = []

for cfg in MODEL_CONFIGS:
    print(f"\n=== Evaluating {cfg['name']} ===")
    try:
        model, ckpt_path = load_model(cfg)
        mode = 'resnet' if 'resnet' in cfg['type'] else 'grayscale'
        ds = EvalDataset(IMG_DIR, LBL_DIR, mode=mode)
        dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

        scores, labels = infer_scores(model, dl)
        thr = threshold_from_normals(scores, labels, target_fpr=TARGET_FPR)
        metrics = compute_metrics(labels, scores, thr)

        plot_and_save_cm(labels, scores, thr, cfg['name'], cfg['key'])
        fpr_vals, tpr_vals, auroc_val = plot_and_save_roc(labels, scores, cfg['name'], cfg['key'])
        roc_curves.append((cfg['name'], fpr_vals, tpr_vals, auroc_val))

        row = {'Model': cfg['name'], 'Checkpoint': str(ckpt_path), 'ThresholdMethod': f'FPR@{TARGET_FPR:.2f} (normal-only)'}
        row.update({
            'Threshold': metrics['threshold'],
            'AUROC': metrics['auroc'],
            'AUPRC': metrics['auprc'],
            'Accuracy': metrics['accuracy'],
            'Precision': metrics['precision'],
            'Recall': metrics['recall'],
            'Specificity': metrics['specificity'],
            'F1': metrics['f1_score'],
            'FPR': metrics['fpr'],
            'FNR': metrics['fnr'],
            'TP': metrics['tp'], 'TN': metrics['tn'], 'FP': metrics['fp'], 'FN': metrics['fn'],
            'N': int(len(labels))
        })
        rows.append(row)
        print(f"AUROC={metrics['auroc']:.4f} | Recall={metrics['recall']:.4f} | F1={metrics['f1_score']:.4f}")
    except Exception as e:
        print(f"FAILED: {cfg['name']} -> {e}")

# Combined ROC
if len(roc_curves) > 0:
    fig, ax = plt.subplots(figsize=(8,7))
    for name, fpr_vals, tpr_vals, auroc_val in roc_curves:
        ax.plot(fpr_vals, tpr_vals, lw=2, label=f"{name} (AUROC={auroc_val:.3f})")
    ax.plot([0,1],[0,1],'k--',lw=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Comparison - Test with Segmentation Labels')
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT_FIG_DIR / 'all_models_roc_comparison.png', dpi=150)
    plt.close(fig)

# Save metrics
if len(rows) > 0:
    df = pd.DataFrame(rows).sort_values('AUROC', ascending=False).reset_index(drop=True)
    df.to_csv(OUT_CSV, index=False)
    display(df)
    print(f"\nMetrics saved: {OUT_CSV}")
    print(f"Figures saved: {OUT_FIG_DIR}")
else:
    print('No model was evaluated successfully.')